# 缺失、重复和异常数据应该怎样处理？：参考答案

每道题先写预期结果再运行。练习中的 TODO 需要自行完成；能顺序运行不代表已完成练习。

In [1]:
from pathlib import Path
CASE_DIR = Path.cwd()
# 载入本例的可修改函数和数据，所有算法都在同目录 solution.py。
exec(compile((CASE_DIR / "solution.py").read_text(encoding="utf-8"), str(CASE_DIR / "solution.py"), "exec"))

                 time  temperature
0 2024-07-01 00:00:00         20.0
1 2024-07-01 01:00:00         21.0
2 2024-07-01 01:00:00         22.0
3 2024-07-01 02:00:00          NaN
4 2024-07-01 03:00:00         24.0
5 2024-07-01 04:00:00        999.0
6 2024-07-01 05:00:00          NaN
7 2024-07-01 06:00:00         26.0
{'input_rows': 8, 'invalid_time': 0, 'unparseable_or_missing': 2, 'duplicate_rows': 1, 'range_violations': 1, 'remaining_missing': 1, 'output_rows': 6}


## 1. 补全：数出被填充的点
仅统计原先缺失且现在非空的记录，并核对本例为 2 个。

In [2]:
filled = work['was_missing'] & work['temperature_clean'].notna()
assert int(filled.sum()) == 2
print(work.loc[filled, ['time', 'temperature_clean']])

                 time  temperature_clean
3 2024-07-01 02:00:00               22.0
5 2024-07-01 04:00:00               24.0


## 2. 修改：允许连续填两点
比较 limit=1 与 limit=2，确认缺失数量不增加，并解释为什么更多填充不一定更好。

In [3]:
longer = work['temperature'].ffill(limit=2)
assert longer.isna().sum() <= work['temperature_clean'].isna().sum()
print('多填充点数', int(longer.notna().sum() - work['temperature_clean'].notna().sum()))

多填充点数 1


## 3. 迁移：湿度的物理范围
为湿度 [45, -2, 110, 60, None] 设置 0—100 的有效范围；不要沿用温度规则。

In [4]:
humidity = pd.Series([45, -2, 110, 60, None], dtype=float)
valid = humidity.where(humidity.between(0, 100))
assert valid.notna().sum() == 2
assert valid.dropna().tolist() == [45, 60]
print(valid)

0    45.0
1     NaN
2     NaN
3    60.0
4     NaN
dtype: float64
